In [ ]:
from netmiko import ConnectHandler, redispatch
import time
import platform
import os
import pandas as pd
import re

server_ip='192.168.218.204'

ip='192.168.218.181'

device_username='student1'
password='Passw0rd1'

def ping_device(ip):
    # Check the platform type
    param = '-n' if platform.system().lower() == 'windows' else '-c'
    null = 'nul' if platform.system().lower() == 'windows' else '/dev/null'
    # Construct and run the command
    response = os.system(f"ping {param} 1 {ip} > {null} 2>&1")
    return response == 0

if not ping_device(ip):
    print(f'ip:{ip}, ping:False, sso_ssh:False, device_ssh:False, device_type:False\n')
else:
    print(f'ip:{ip}, ping:True')
    print('Connecting to SSO server')
    try:
        server = {
            'device_type': 'terminal_server',
            'host': server_ip,
            'username': device_username,
            'password': password,
            'port': 2222,
            'conn_timeout': 80,
            'global_delay_factor': 40,
            'session_log': 'outputx.log'
        }

        jump_conn = ConnectHandler(**server)
        print('servera baglandik')
    except Exception as e:
        print(e)
        print(f'ip:{ip},sso_ssh:False, device_ssh:False, device_type:False\n')

    else:
        try:
            print('try')

            jump_conn.read_until_pattern(pattern='search or select one:',
                                         read_timeout=40)

            print(f'>> IP gönderiliyor: {ip}')
            jump_conn.write_channel(f"{ip}\n")

            jump_conn.read_until_pattern(pattern=r"[Pp]assword:",read_timeout=50)

            print('>> Cihaz şifresi gönderiliyor')
            jump_conn.write_channel(f"{server['password']}\n")

            prompt_out = jump_conn.read_until_pattern(pattern=r"[>#]",read_timeout=50)
            print('>> Prompt geldi:')
            #print(prompt_out)

            redispatch(jump_conn, device_type='autodetect')

            output=jump_conn.send_command('show interfaces',
                                          delay_factor=10)
            print(output)

        except Exception as e:
            print(f'ip:{ip},sso_ssh:True, device_ssh:False, device_type:False')
            print(e)
        finally:
            jump_conn.disconnect()


In [ ]:
import os
import platform
from netmiko import ConnectHandler, redispatch

server_ip = '192.168.218.204'
ip = '192.168.218.181'
device_username = 'student1'
decrypted_password = 'Passw0rd1'

def ping_device(target_ip):
    param = '-n' if platform.system().lower() == 'windows' else '-c'
    null = 'nul' if platform.system().lower() == 'windows' else '/dev/null'
    return os.system(f"ping {param} 1 {target_ip} > {null} 2>&1") == 0

if not ping_device(ip):
    print(f'ip:{ip}, ping:False, sso_ssh:False, device_ssh:False')
else:
    print(f'ip:{ip}, ping:True')
    print('Connecting to SSO server...')
    try:
        server = {
            'device_type': 'terminal_server',
            'host': server_ip,
            'username': device_username,
            'password': decrypted_password,
            'port': 2222,
            'conn_timeout': 30,
            'global_delay_factor': 2,
            'session_log': 'outputx.log'
        }
        jump_conn = ConnectHandler(**server)
        print('SSO Sunucusuna baglanildi.')

        jump_conn.read_until_pattern(pattern='search or select one:', read_timeout=30)
        print(f'>> IP gonderiliyor: {ip}')
        jump_conn.write_channel(f"{ip}\n")

        jump_conn.read_until_pattern(pattern=r"[Pp]assword:", read_timeout=30)
        print('>> Cihaz sifresi gonderiliyor...')
        jump_conn.write_channel(f"{decrypted_password}\n")

        prompt_out = jump_conn.read_until_pattern(pattern=r"[>#]", read_timeout=30)
        print(f'>> Prompt alindi: {prompt_out.strip()}')

        redispatch(jump_conn, device_type='cisco_ios')
        
        output = jump_conn.send_command('show ip interface brief')
        print("\n--- Cihaz Ciktisi ---")
        print(output)
        
    except Exception as e:
        print(f'Hata: {e}')
    finally:
        if 'jump_conn' in locals():
            jump_conn.disconnect()

In [1]:
from netmiko import ConnectHandler, redispatch
import time
import platform
import os
import re
from multiprocessing.dummy import Pool as ThreadPool

server_ip='192.168.218.204'

results=[]

devices=['192.168.218.181','192.168.218.182','192.168.218.183','192.168.218.184']

IDENTIFY_COMMANDS=['show version','display version']

user = 'student1'
sifre = 'Passw0rd1'      # sen ne kullanıyorsan

def ping_device(ip):
    param = '-n' if platform.system().lower() == 'windows' else '-c'
    null = 'nul' if platform.system().lower() == 'windows' else '/dev/null'
    response = os.system(f"ping {param} 1 {ip} > {null} 2>&1")
    return response == 0

def device_type_detect(ip):
    if not ping_device(ip):
        results.append({'ip':ip, 'ping':False, 'sso_ssh':False, 'device_ssh':False, 'device_type':False})
        print(f'ip:{ip}, ping:False, sso_ssh:False, device_ssh:False, device_type:False\n')
        return

    print(f'ip:{ip}, ping:True')
    print('Connecting to SSO server')
    try:
        server = {
            'device_type': 'terminal_server',
            'host': server_ip,
            'username': user,
            'password': sifre,
            'port': 2222,
            'conn_timeout': 80,
            'global_delay_factor': 40,
            'session_log': 'outputx.log'
        }

        jump_conn = ConnectHandler(**server)
        print('servera baglandik')
    except Exception as e:
        results.append({'ip':ip,
                         'ping':True,
                         'sso_ssh':False,
                         'device_ssh':False,
                         'device_type':False})

        print(e)
        print(f'ip:{ip},sso_ssh:False, device_ssh:False, device_type:False\n')
        return
    #burada kaldik 13:00 oglen molasi sonrasi
    else:
        try:
            print('try')

            jump_conn.read_until_pattern(pattern='search or select one:',
                                         read_timeout=40)

            print(f'>> IP gönderiliyor: {ip}')
            jump_conn.write_channel(f"{ip}\n")

            jump_conn.read_until_pattern(pattern=r"[Pp]assword:",read_timeout=50)

            print('>> Cihaz şifresi gönderiliyor')
            jump_conn.write_channel(f"{server['password']}\n")

            prompt_out = jump_conn.read_until_pattern(pattern=r"[>#]",read_timeout=50)
            print('>> Prompt geldi:')
            print(prompt_out)

            redispatch(jump_conn, device_type='autodetect')

            device_type = None
            for cmd in IDENTIFY_COMMANDS:
                print(f'input: {cmd}\n')
                jump_conn.write_channel(f'{cmd}\n')
                time.sleep(1)
                output=jump_conn.read_channel()

                if 'Huawei' in output or 'HUAWEI' in output:
                    device_type = 'huawei'
                    break
                elif 'Cisco' in output:
                    if 'Cisco IOS-XE software,' in output or 'IOS-XE ROMMON' in output:
                        device_type = 'cisco_xe'
                        break
                    elif 'Cisco IOS Software' in output:
                        device_type = 'cisco_ios'
                        break
                    elif 'Cisco Nexus Operating System' in output:
                        device_type = 'cisco_nxos'
                        break
                    elif 'Cisco IOS XR Software' in output:
                        device_type = 'cisco_xr'
                        break

                elif 'Arista' in output:
                    device_type = 'arista_eos'
                    break

            if device_type:
                results.append({'ip':ip,
                 'ping':True,
                 'sso_ssh':True,
                 'device_ssh':True,
                 'device_type':device_type})

                print(f'ip:{ip},sso_ssh:True, device_ssh:True, device_type:{device_type}\n')

            if not device_type:
                results.append({'ip':ip,
                 'ping':True,
                 'sso_ssh':True,
                 'device_ssh':True,
                 'device_type':False})
                print(f'ip:{ip}, sso_ssh:True, device_ssh:True, device_type:False\n')

        except Exception as e:

            results.append({'ip':ip,
                 'ping':True,
                 'sso_ssh':True,
                 'device_ssh':False,
                 'device_type':False})
            print(f'ip:{ip},sso_ssh:True, device_ssh:False, device_type:False')
            print(e)
        finally:
            jump_conn.disconnect()


myPool = ThreadPool(4)
result = myPool.map(device_type_detect, devices)


ip:192.168.218.182, ping:True
Connecting to SSO server
ip:192.168.218.183, ping:True
Connecting to SSO server
ip:192.168.218.181, ping:True
Connecting to SSO server
ip:192.168.218.184, ping:True
Connecting to SSO server
servera baglandik
try
>> IP gönderiliyor: 192.168.218.183
servera baglandik
try
>> IP gönderiliyor: 192.168.218.181
>> Cihaz şifresi gönderiliyor
servera baglandik
try
>> IP gönderiliyor: 192.168.218.182
>> Cihaz şifresi gönderiliyor
>> Prompt geldi:
 

SW1#
input: show version

>> Cihaz şifresi gönderiliyor
>> Prompt geldi:
 

SW2#
servera baglandik
try
>> IP gönderiliyor: 192.168.218.184
input: show version

>> Prompt geldi:
 
Last login: Thu Sep 24 17:29:47 2026
arista>
input: show version

ip:192.168.218.181,sso_ssh:True, device_ssh:True, device_type:cisco_ios

ip:192.168.218.182,sso_ssh:True, device_ssh:True, device_type:cisco_ios

ip:192.168.218.183,sso_ssh:True, device_ssh:True, device_type:arista_eos

>> Cihaz şifresi gönderiliyor
>> Prompt geldi:
 

  ---------

In [2]:
results

[{'ip': '192.168.218.181',
  'ping': True,
  'sso_ssh': True,
  'device_ssh': True,
  'device_type': 'cisco_ios'},
 {'ip': '192.168.218.182',
  'ping': True,
  'sso_ssh': True,
  'device_ssh': True,
  'device_type': 'cisco_ios'},
 {'ip': '192.168.218.183',
  'ping': True,
  'sso_ssh': True,
  'device_ssh': True,
  'device_type': 'arista_eos'},
 {'ip': '192.168.218.184',
  'ping': True,
  'sso_ssh': True,
  'device_ssh': True,
  'device_type': 'huawei'}]